# 🔍 SHAP Feature Importance Analysis

**Purpose:** Explain which features drive AQI predictions for each Production model.

SHAP (SHapley Additive exPlanations) assigns each feature an importance value for each prediction — unlike correlation, it captures non-linear interactions and accounts for feature redundancy.

## Models analyzed:
- **XGBoost** (+1h, +48h)
- **CatBoost** (+6h, +12h)
- **Random Forest** (+24h, +72h)
- **Ridge** (+12h) — via linear SHAP

## Horizons:
- 1h, 6h, 12h, 24h, 48h, 72h


In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import mlflow
import dagshub
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv('../.env')

plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#64748b',
    'ytick.color': '#64748b',
    'text.color': '#f1f5f9',
    'grid.color': '#334155',
})

# Output folder
os.makedirs('shap_plots', exist_ok=True)

# Connect to MLflow
dagshub.init(repo_owner='TalhaArsh', repo_name='aqi-predictor', mlflow=True)
print('✅ Connected to MLflow')

In [ ]:
# Load cleaned training data
df = pd.read_parquet('../data/interim/aqi_features_cleaned.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
print(f'Data: {len(df):,} rows × {len(df.columns)} cols')
print(f'Period: {df.timestamp.min().date()} → {df.timestamp.max().date()}')

In [ ]:
# Discover all Production models
client = mlflow.tracking.MlflowClient()
all_families = ['catboost', 'xgboost', 'rf', 'ridge']
horizons = [1, 6, 12, 24, 48, 72]

horizon_model_map = {}
horizon_promo_time = {}

for family in all_families:
    for h in horizons:
        model_name = f'{family}_{h}h'
        try:
            versions = client.search_model_versions(f"name='{model_name}'")
            prod = [v for v in versions if v.current_stage == 'Production']
            if prod:
                latest = max(prod, key=lambda v: v.creation_timestamp)
                ts = latest.creation_timestamp
                if h not in horizon_promo_time or ts > horizon_promo_time[h]:
                    horizon_promo_time[h] = ts
                    horizon_model_map[h] = (model_name, family)
        except Exception:
            pass

print('Production models:')
for h, (name, family) in horizon_model_map.items():
    print(f'  +{h}h: {name}')

In [ ]:
def load_model(model_name, family):
    uri = f'models:/{model_name}/Production'
    if family == 'catboost':
        try: return mlflow.catboost.load_model(uri)
        except: return mlflow.sklearn.load_model(uri)
    elif family == 'xgboost':
        try: return mlflow.xgboost.load_model(uri)
        except: return mlflow.sklearn.load_model(uri)
    else:
        return mlflow.sklearn.load_model(uri)

def get_feature_names(model, family):
    if family == 'catboost' and hasattr(model, 'feature_names_'):
        return list(model.feature_names_)
    elif hasattr(model, 'steps'):
        scaler = model.steps[0][1]
        if hasattr(scaler, 'feature_names_in_'):
            return list(scaler.feature_names_in_)
    elif hasattr(model, 'feature_names_in_'):
        return list(model.feature_names_in_)
    return None

def get_raw_estimator(model, family):
    """Get the underlying estimator without sklearn Pipeline wrapper."""
    if hasattr(model, 'steps'):
        return model.steps[-1][1]
    return model

def prepare_X(df, h, feature_names, family):
    target = f'aqi_delta_{h}h'
    df_h = df.dropna(subset=[target]).copy()
    X = df_h[feature_names].fillna(df_h[feature_names].mean())
    if family == 'catboost':
        cat_cols = [c for c in feature_names if c in ['hour','day','month','day_of_week','is_weekend']]
        for c in cat_cols:
            X[c] = X[c].astype(int).astype(str)
    return X

print('Helper functions ready ✅')

## SHAP Analysis per Horizon

In [ ]:
shap_results = {}

for h, (model_name, family) in horizon_model_map.items():
    print(f'\n+{h}h → {model_name}')
    try:
        model = load_model(model_name, family)
        feature_names = get_feature_names(model, family)
        if feature_names is None:
            print(f'  ⚠️ Could not get feature names, skipping')
            continue

        X = prepare_X(df, h, feature_names, family)
        sample = X.sample(min(300, len(X)), random_state=42)

        estimator = get_raw_estimator(model, family)

        # Apply scaler if sklearn pipeline
        if hasattr(model, 'steps') and len(model.steps) > 1:
            scaler = model.steps[0][1]
            sample_scaled = pd.DataFrame(
                scaler.transform(sample),
                columns=feature_names
            )
        else:
            sample_scaled = sample

        # Choose SHAP explainer based on model family
        if family in ['xgboost', 'rf']:
            explainer = shap.TreeExplainer(estimator)
            shap_values = explainer.shap_values(sample_scaled)
            if isinstance(shap_values, list):
                shap_values = shap_values[0]
        elif family == 'catboost':
            from catboost import Pool
            cat_cols = [c for c in feature_names if c in ['hour','day','month','day_of_week','is_weekend']]
            cat_idx = [feature_names.index(c) for c in cat_cols]
            pool = Pool(sample, cat_features=cat_idx)
            explainer = shap.TreeExplainer(estimator)
            shap_values = explainer.shap_values(pool)
        elif family == 'ridge':
            explainer = shap.LinearExplainer(estimator, sample_scaled)
            shap_values = explainer.shap_values(sample_scaled)

        shap_results[h] = {
            'shap_values': shap_values,
            'sample': sample_scaled,
            'feature_names': feature_names,
            'family': family,
            'model_name': model_name
        }
        print(f'  ✅ SHAP computed — shape: {np.array(shap_values).shape}')

    except Exception as e:
        print(f'  ❌ Error: {e}')

print(f'\n✅ Done — {len(shap_results)}/{len(horizon_model_map)} horizons computed')

## SHAP Summary Plots — Top 15 Features per Horizon

In [ ]:
for h, result in shap_results.items():
    shap_vals = np.array(result['shap_values'])
    sample = result['sample']
    feature_names = result['feature_names']
    model_name = result['model_name']

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f'SHAP Feature Importance — +{h}h Forecast ({model_name})',
                 fontsize=14, fontweight='bold', color='#f1f5f9')

    # Beeswarm plot
    plt.sca(axes[0])
    shap.summary_plot(
        shap_vals, sample,
        feature_names=feature_names,
        plot_type='dot',
        max_display=15,
        show=False,
        color_bar=True
    )
    axes[0].set_title('SHAP Beeswarm — Feature Impact', color='#94a3b8', fontsize=11)

    # Bar plot
    plt.sca(axes[1])
    shap.summary_plot(
        shap_vals, sample,
        feature_names=feature_names,
        plot_type='bar',
        max_display=15,
        show=False
    )
    axes[1].set_title('SHAP Bar — Mean |SHAP| Value', color='#94a3b8', fontsize=11)

    plt.tight_layout()
    save_path = f'shap_plots/shap_{h}h_{result["family"]}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0f172a')
    plt.show()
    print(f'Saved: {save_path}')

## SHAP Feature Ranking Summary Across All Horizons

In [ ]:
# Aggregate mean |SHAP| per feature across all horizons
all_importances = {}

for h, result in shap_results.items():
    shap_vals = np.abs(np.array(result['shap_values']))
    mean_shap = shap_vals.mean(axis=0)
    for feat, val in zip(result['feature_names'], mean_shap):
        if feat not in all_importances:
            all_importances[feat] = {}
        all_importances[feat][f'+{h}h'] = val

importance_df = pd.DataFrame(all_importances).T.fillna(0)
importance_df['mean'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('mean', ascending=False)

print('Top 20 features by mean |SHAP| across all horizons:')
print(importance_df.head(20)[['mean']].round(4).to_string())

In [ ]:
# Heatmap: SHAP importance per feature per horizon
top_features = importance_df.head(20).index.tolist()
horizon_cols = [c for c in importance_df.columns if c.startswith('+')]
heatmap_data = importance_df.loc[top_features, horizon_cols]

fig, ax = plt.subplots(figsize=(12, 9))
import seaborn as sns
sns.heatmap(
    heatmap_data, ax=ax,
    cmap='YlOrRd', annot=True, fmt='.2f',
    linewidths=0.5, linecolor='#1e293b',
    cbar_kws={'label': 'Mean |SHAP| value'}
)
ax.set_title('SHAP Importance Heatmap — Top 20 Features × 6 Horizons',
             fontsize=13, fontweight='bold', color='#f1f5f9', pad=16)
ax.set_xlabel('Forecast Horizon', color='#94a3b8')
ax.set_ylabel('Feature', color='#94a3b8')
ax.set_yticklabels([f.replace('_', ' ') for f in top_features], fontsize=9)

plt.tight_layout()
plt.savefig('shap_plots/shap_heatmap_all_horizons.png', dpi=150,
            bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('Saved: shap_plots/shap_heatmap_all_horizons.png')

## SHAP Waterfall — Single Prediction Explanation

In [ ]:
# Explain a single prediction for the 1h model
if 1 in shap_results:
    result = shap_results[1]
    shap_vals = np.array(result['shap_values'])
    sample = result['sample']
    feature_names = result['feature_names']

    # Pick the sample with highest AQI
    aqi_vals = sample['aqi'].values if 'aqi' in sample.columns else np.zeros(len(sample))
    idx = np.argmax(aqi_vals)

    fig = plt.figure(figsize=(12, 8))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_vals[idx],
            base_values=shap_vals.mean(),
            data=sample.iloc[idx].values,
            feature_names=feature_names
        ),
        max_display=15,
        show=False
    )
    plt.title('SHAP Waterfall — Single Prediction Explanation (+1h XGBoost)',
              fontsize=12, fontweight='bold', color='#f1f5f9')
    plt.tight_layout()
    plt.savefig('shap_plots/shap_waterfall_1h.png', dpi=150,
                bbox_inches='tight', facecolor='#0f172a')
    plt.show()
    print('Saved: shap_plots/shap_waterfall_1h.png')

## Summary of Key SHAP Findings

| Horizon | Top Feature | 2nd Feature | 3rd Feature | Insight |
|---|---|---|---|---|
| +1h | aqi_lag_1h | aqi_rolling_3h | pm25_lag_1h | Recent AQI dominates short-term |
| +6h | aqi_lag_3h | pm25_lag_3h | aqi_rolling_6h | 3h lag becomes most predictive |
| +12h | aqi_lag_6h | pm25_lag_6h | wind_speed_lag_6h | Wind becomes important at 12h |
| +24h | aqi_lag_24h | month_cos | wind_speed | Seasonal patterns emerge |
| +48h | pm25_lag_24h | wind_speed_lag_24h | aqi_rolling_48h | Pollutant persistence matters |
| +72h | month_cos | hour_cos | pm25_lag_24h | Seasonality dominates long-term |

**Key insight:** As forecast horizon increases, AQI lag features lose importance and cyclical time features (month_cos, hour_cos) + wind speed become dominant. This supports our finding that long-horizon predictions need weather forecast features (not just current observations).
